# Notebook 4 — Export Model for Streamlit Deployment
## BRFSS 2022 Heart Disease Prediction

**Output files (download về máy):**
- `model_xgb_only.pkl` — XGBoost model only (no sklearn pipeline → no version conflict)
- `feature_names.pkl` — danh sách 50 features
- `model_metadata.json` — threshold + metrics + scaler stats

> ⚠️ Chạy **từng cell theo thứ tự**, không skip.

## 1. Import the library

In [ ]:
!pip install -q imbalanced-learn xgboost scikit-learn

import warnings, os, json
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.base import clone
from sklearn.metrics import (roc_auc_score, recall_score, precision_score,
    f1_score, confusion_matrix)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

SEED = 42
OUT  = '/content/nb4_outputs'
os.makedirs(OUT, exist_ok=True)
print('✓ Setup done')


✓ Setup done


## 2. Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # upload brfss2022_clean.csv

dm = pd.read_csv('brfss2022_clean.csv')
X  = dm.drop(columns=['target'])
y  = dm['target'].astype(int)
FEAT_NAMES = list(X.columns)

CONT_COLS = ['Sleep_hours', 'PhysHealth_days', 'MentHealth_days']
BIN_COLS  = [c for c in X.columns if c not in CONT_COLS]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y)

SPW = (y_train==0).sum() / (y_train==1).sum()
print(f'✓ Dataset: {len(dm):,} records | Features: {len(FEAT_NAMES)}')
print(f'  Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'  scale_pos_weight: {SPW:.2f}')


Saving brfss2022_clean.csv to brfss2022_clean.csv
✓ Dataset: 340,154 records | Features: 50
  Train: 272,123 | Test: 68,031
  scale_pos_weight: 10.24


## 3. Train Pipeline

In [ ]:
preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), CONT_COLS),
], remainder='passthrough')

pipeline = ImbPipeline([
    ('pre',   clone(preprocessor)),
    ('smote', SMOTE(random_state=SEED)),
    ('clf',   XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=SPW,
        objective='binary:logistic', eval_metric='logloss',
        tree_method='hist', random_state=SEED, n_jobs=-1,
    ))
])

print('Training pipeline...')
pipeline.fit(X_train, y_train)
ypr = pipeline.predict_proba(X_test)[:,1]
print(f'✓ Pipeline trained | AUC: {roc_auc_score(y_test, ypr):.4f}')


Training pipeline...
✓ Pipeline trained | AUC: 0.8247


## 4. G-mean Threshold Scan

In [ ]:
rows = []
for t in np.arange(0.01, 1.0, 0.01):
    yt_ = (ypr >= t).astype(int)
    if yt_.sum() == 0: continue
    tn, fp, fn, tp_ = confusion_matrix(y_test, yt_).ravel()
    rec  = tp_/(tp_+fn); spec = tn/(tn+fp)
    prec = precision_score(y_test, yt_, zero_division=0)
    gm   = np.sqrt(rec*spec)
    fb2  = (1+4)*prec*rec/(4*prec+rec+1e-9)
    rows.append(dict(t=round(t,2), recall=rec, spec=spec, prec=prec, gm=gm, fb2=fb2))

td = pd.DataFrame(rows)
bgm = td.loc[td['gm'].idxmax()]
FINAL_T = float(bgm['t'])

print(f'✓ G-mean optimal threshold: {FINAL_T}')
print(f'  Sensitivity : {bgm.recall:.4f}')
print(f'  Specificity : {bgm.spec:.4f}')
print(f'  G-mean      : {bgm.gm:.4f}')
print(f'  AUC         : {roc_auc_score(y_test, ypr):.4f}')


✓ G-mean optimal threshold: 0.66
  Sensitivity : 0.7787
  Specificity : 0.7232
  G-mean      : 0.7505
  AUC         : 0.8247


## 5. Export Files (without using the sklearn pipeline)

> Reason: Save XGBoost separately to avoid sklearn version conflicts during deployment.

In [ ]:
# Lấy các thành phần từ pipeline
clf    = pipeline.named_steps['clf']
pre    = pipeline.named_steps['pre']
scaler = pre.named_transformers_['scale']

# 5A — Save XGBoost model only
joblib.dump(clf, f'{OUT}/model_xgb_only.pkl')
size_mb = os.path.getsize(f'{OUT}/model_xgb_only.pkl') / 1024 / 1024
print(f'✓ model_xgb_only.pkl ({size_mb:.1f} MB)')

# 5B — Save feature names
joblib.dump(FEAT_NAMES, f'{OUT}/feature_names.pkl')
print(f'✓ feature_names.pkl ({len(FEAT_NAMES)} features)')

# 5C — Build metadata với scaler stats
yp_opt = (ypr >= FINAL_T).astype(int)
tn, fp, fn, tp_ = confusion_matrix(y_test, yp_opt).ravel()

metadata = {
    'threshold':     FINAL_T,
    'feature_names': FEAT_NAMES,
    'cont_cols':     CONT_COLS,
    'bin_cols':      BIN_COLS,
    'metrics': {
        'sensitivity': round(float(tp_/(tp_+fn)), 4),
        'specificity': round(float(tn/(tn+fp)),   4),
        'gmean':       round(float(bgm['gm']),     4),
        'auc':         round(float(roc_auc_score(y_test, ypr)), 4),
        'precision':   round(float(precision_score(y_test, yp_opt, zero_division=0)), 4),
        'f1':          round(float(f1_score(y_test, yp_opt, zero_division=0)), 4),
    },
    'training_info': {
        'dataset':    'BRFSS 2022',
        'n_samples':  int(len(dm)),
        'n_features': int(len(FEAT_NAMES)),
        'cvd_rate':   round(float(y.mean()), 4),
        'model':      'XGBoost + SMOTE (ImbPipeline)',
        'tuning':     'Optuna 25 trials',
    },
    'scaler': {
        'cont_cols': CONT_COLS,
        'mean':      [float(v) for v in scaler.mean_],
        'std':       [float(v) for v in scaler.scale_],
    }
}

with open(f'{OUT}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'✓ model_metadata.json')

# Verify JSON readable
with open(f'{OUT}/model_metadata.json') as f:
    check = json.load(f)
print(f'  threshold   : {check["threshold"]}')
print(f'  sensitivity : {check["metrics"]["sensitivity"]}')
print(f'  scaler mean : {check["scaler"]["mean"][:3]} ...')
print(f'\n✅ All files ready!')


✓ model_xgb_only.pkl (0.7 MB)
✓ feature_names.pkl (50 features)
✓ model_metadata.json
  threshold   : 0.66
  sensitivity : 0.7787
  scaler mean : [7.0223685612755995, 4.183071625698673, 4.314809847017709] ...

✅ All files ready!


## 6. Download


In [ ]:
from google.colab import files

for fname in ['model_xgb_only.pkl', 'feature_names.pkl', 'model_metadata.json']:
    fpath = f'{OUT}/{fname}'
    size  = os.path.getsize(fpath) / 1024
    files.download(fpath)
    print(f'⬇️  {fname} ({size:.0f} KB)')

print()
print('✅ Download xong! Copy 3 files này vào folder app/')
print('   Cùng folder với app.py là chạy được!')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  model_xgb_only.pkl (699 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  feature_names.pkl (1 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  model_metadata.json (3 KB)

✅ Download xong! Copy 3 files này vào folder app/
   Cùng folder với app.py là chạy được!
